In [1]:
import pandas as pd
import torch
from utils import loadVocab, START_TAG, END_TAG, UNK_TAG,vocab_path

In [2]:
#Import module from CDCR folder
import sys
sys.path.append(r'C:\Users\sago\Documents\GitHub\CDCR_lexical_diversity\CDCR\data')

import dataset

In [3]:
ecbplus_config = dataset.DatasetConfig(data_folder="../../CDCR/data/ECBplus-prep/output_data/",mention_type="entities_mentions",use_gold_mentions=True)

In [4]:
ecbplus_train = dataset.CrossEncoderDatasetFull(config=ecbplus_config,split_name='train')
ecbplus_test = dataset.CrossEncoderDatasetFull(config=ecbplus_config,split_name='test')
ecbplus_val = dataset.CrossEncoderDatasetFull(config=ecbplus_config,split_name='val')

In [12]:
#Modification from the original load_text_con method. Adds start and end tags
def load_text_con(cdcr_dataset_split,max_rows=None):
    if max_rows==None:
        max_rows=len(cdcr_dataset_split.instances)
    #Load CDCR data
    
    inputs = [[START_TAG] + x.strip().split(" ") + [END_TAG] for x in cdcr_dataset_split.instances[0:max_rows]] 
    # I could add this to the standard processing. There is unnecessary overhead
    

    # label_with_pos = [[START_TAG] + x + [SEP_TAG] + y + [END_TAG] for x, y in zip(label_sen, label_pos)]
    labels = cdcr_dataset_split.labels[0:max_rows]

    return inputs, labels

In [13]:
#Temp vocab based on MSCOCO. I should use a pre-trained subtokenizer
#loadVocab(vocab_fpath="mscoco.vocab")

In [18]:
def load_data_tensor_con(cdcr_dataset,max_rows, vocabPath, maxlen=300): #  have to adapt the max length become the max length of the input dataset
    token2idx, _ = loadVocab(vocabPath)

    tokenized_inputs = []
    inputs, labels = load_text_con(cdcr_dataset,max_rows)
    padded_length = maxlen
    for x in inputs:
        x = [token2idx.get(t, token2idx[UNK_TAG]) for t in x] #Fills with Unknown tag value if token not found 
        # Definitely need to improve the tokenization. Gotta be careful with how to add these values to the vocabulary
        x = x + [0] * (padded_length - len(x)) 

        tokenized_inputs.append(x)
    return torch.tensor(tokenized_inputs), labels

In [19]:
ecbplus_train_tokenized_inputs, ecbplus_train_labels = load_data_tensor_con(cdcr_dataset=ecbplus_train,max_rows=None,vocabPath=vocab_path)
ecbplus_test_tokenized_inputs, ecbplus_test_labels = load_data_tensor_con(cdcr_dataset=ecbplus_test,max_rows=None,vocabPath=vocab_path)
ecbplus_val_tokenized_inputs, ecbplus_val_labels = load_data_tensor_con(cdcr_dataset=ecbplus_test,max_rows=None,vocabPath=vocab_path)


In [27]:

torch.save(ecbplus_train_tokenized_inputs, 'train_inputs.pt')
torch.save(ecbplus_train_labels, 'train_labels.pt')

torch.save(ecbplus_test_tokenized_inputs, 'test_inputs.pt')
torch.save(ecbplus_test_labels, 'test_labels.pt')

torch.save(ecbplus_val_tokenized_inputs, 'val_inputs.pt')
torch.save(ecbplus_val_labels, 'val_labels.pt')



In [ ]:

# (Zeng et al., 2020) Suggest oversampling positive pairs and undersampling negative pairs